In [1]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
from scipy import stats
import sys
import seaborn as sns
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *
import TA_tools
import importlib
importlib.reload(TA_tools)

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
SS_VER = 'SS_VER_1_1'

The purpose of this experiment is to determine if for each of the four possible inputs, traces are similar across each butterfly operation. If so great, if not, how different are they?

In [2]:
%run "/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/adversary/TA_segmented_initialise.ipynb"

INFO: Found ChipWhisperer😍


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:397) Could not adjust adc_mul via output divider alone. Recalcing clocks...
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:398) Target clock has dropped for a moment. You may need to reset your target


scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [3]:
tools = TA_tools.Tools(scope,target)

In [4]:
x0 = "00000000\n"  #00
x1 = "00001000\n"  #01
x2 = "10000000\n"  #10
x3 = "10001000\n"  #11

dummy = "11111111\n"
warmup = tools.get_trace(dummy)

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:695) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid


RuntimeError: Capture failed

In [ ]:
cw .plot(warmup[0][0:4624]) * cw.plot(warmup[1][0:4624])

In [ ]:
pattern = warmup[0][900:1500]
warmup_aligned = tools.align_traces(pattern, warmup)
np.save("pattern.npy", pattern)

In [ ]:
cw.plot(warmup_aligned[0][0:4624]) * cw.plot(warmup_aligned[1][0:4624]) * cw.plot(warmup[0][0:4624]+ 0.8) * cw.plot(warmup[1][0:4624] + 0.8)

In [ ]:
t1 = []
t2 = []
t3 = []
t4 = []
for i in range(10):
    t1.append(tools.get_trace(x0)[0:3920])
    t2.append(tools.get_trace(x1)[0:3920])
    t3.append(tools.get_trace(x2)[0:3920])
    t4.append(tools.get_trace(x3)[0:3920])
t1 = np.array(t1)
t2 = np.array(t2)
t3 = np.array(t3)
t4 = np.array(t4)

t1 = tools.allign_traces_vec(pattern, t1)
t2 = tools.allign_traces_vec(pattern, t2)
t3 = tools.allign_traces_vec(pattern, t3)
t4 = tools.allign_traces_vec(pattern, t4)

In [ ]:
cw.plot(t1[0][0]) * cw.plot(t1[0][1]) * cw.plot(t1[0][0] + 0.8) * cw.plot(t1[1][0] + 0.8) * cw.plot(t2[0][0]) * cw.plot(t2[0][0] + 0.8)

In [ ]:
cw.plot(t1[0][0] - t1[0][1]) * cw.plot(t1[0][0] - t4[0][0])

In [ ]:
var_t1_internal_bf1 = np.var(t1[:,0],axis=0)
var_t2_internal_bf1 = np.var(t2[:,0],axis=0)
var_t3_internal_bf1 = np.var(t3[:,0],axis=0)
var_t4_internal_bf1 = np.var(t4[:,0],axis=0)

print("T1 BF1 Internal Var Sum: ", np.sum(var_t1_internal_bf1))
print("T2 BF1 Internal Var Sum: ", np.sum(var_t2_internal_bf1))
print("T3 BF1 Internal Var Sum: ", np.sum(var_t3_internal_bf1))
print("T4 BF1 Internal Var Sum: ", np.sum(var_t4_internal_bf1))

In [ ]:
var_t1_t2_bf1 = np.var(np.vstack((t1[:,0],t2[:,0])),axis=0)
var_t1_t3_bf1 = np.var(np.vstack((t1[:,0],t3[:,0])),axis=0)
var_t1_t4_bf1 = np.var(np.vstack((t1[:,0],t4[:,0])),axis=0)

var_t2_t3_bf1 = np.var(np.vstack((t2[:,0],t3[:,0])),axis=0)
var_t2_t4_bf1 = np.var(np.vstack((t2[:,0],t4[:,0])),axis=0)

var_t3_t4_bf1 = np.var(np.vstack((t3[:,0],t4[:,0])),axis=0)

print("T1-T2 Var Sum: ", np.sum(var_t1_t2_bf1))
print("T1-T3 Var Sum: ", np.sum(var_t1_t3_bf1))
print("T1-T4 Var Sum: ", np.sum(var_t1_t4_bf1))
print("T2-T3 Var Sum: ", np.sum(var_t2_t3_bf1))
print("T2-T4 Var Sum: ", np.sum(var_t2_t4_bf1))
print("T3-T4 Var Sum: ", np.sum(var_t3_t4_bf1))


In [ ]:
cw.plot(var_t1_internal_bf1) * cw.plot(var_t1_t2_bf1+0.002)

In [ ]:
var_t1_internal_bf2 = np.var(t1[:,1],axis=0)
var_t2_internal_bf2 = np.var(t2[:,1],axis=0)
var_t3_internal_bf2 = np.var(t3[:,1],axis=0)
var_t4_internal_bf2 = np.var(t4[:,1],axis=0)

print("T1 BF2 Internal Var Sum: ", np.sum(var_t1_internal_bf2))
print("T2 BF2 Internal Var Sum: ", np.sum(var_t2_internal_bf2))
print("T3 BF2 Internal Var Sum: ", np.sum(var_t3_internal_bf2))
print("T4 BF2 Internal Var Sum: ", np.sum(var_t4_internal_bf2))

In [ ]:
var_t1bf1_t1bf2 = np.var(np.vstack((t1[:,0], t1[:,1])),axis=0)
var_t2bf1_t2bf2 = np.var(np.vstack((t2[:,0], t2[:,1])),axis=0)
var_t3bf1_t3bf2 = np.var(np.vstack((t3[:,0], t3[:,1])),axis=0)
var_t4bf1_t4bf2 = np.var(np.vstack((t4[:,0], t4[:,1])),axis=0)

print("T1 BF1-BF2 Var Sum: ", np.sum(var_t1bf1_t1bf2))
print("T2 BF1-BF2 Var Sum: ", np.sum(var_t2bf1_t2bf2))
print("T3 BF1-BF2 Var Sum: ", np.sum(var_t3bf1_t3bf2))
print("T4 BF1-BF2 Var Sum: ", np.sum(var_t4bf1_t4bf2))

In [ ]:
cw.plot(t2[0][0]- t2[0][1]) * cw.plot(t1[0][0]- t2[0][0])

In [ ]:
cw.plot(t2[0][0]) * cw.plot(t2[0][1]) * cw.plot(t2[0][0] + 0.8) * cw.plot(t2[1][0] + 0.8)# * cw.plot(t1[0][0]) * cw.plot(t1[0][0] + 0.8)

Making the data collection function required for scaling up to a ML model

In [ ]:
def data_collection(repeats, key_length):
    half_n = int(key_length / 2)

    full_set = []
    keys_used = []

    trace_end = 3920

    ### Do all collection at once to save time on re-initialising the scope

    for i in range(repeats):
        #print(f"Data Collection Round: {i+1}/{repeats}")
        correct_key = np.random.randint(2,size=key_length)
        correct_key_str = tools.array_to_bin_input(correct_key)
        correct_key_bit_rev = tools.bit_reverse(correct_key.copy())

        keys_used.append(correct_key_bit_rev.copy())
        full_set.append(tools.get_trace(correct_key_str)[0:trace_end])

        #print(i)

    full_set = tools.allign_traces_vec(pattern, np.array(full_set))

    return full_set, keys_used

def sort_set(repeats, full_set, keys_used, key_length):
    ### Do all sorteing after

    half_n = int(key_length / 2)


    sorted_set = [[[] for _ in range(4)] for _ in range(half_n)] ## n/2 lists for each bf. Subsorted into 4 inputs
    for i in range(repeats):
        for j in range(half_n):
            if (keys_used[i][2*j:2*j+2] == [0,0]).all():
                sorted_set[j][0].append(full_set[i][j])
            elif (keys_used[i][2*j:2*j+2] == [0,1]).all():
                sorted_set[j][1].append(full_set[i][j])
            elif (keys_used[i][2*j:2*j+2] == [1,0]).all():
                sorted_set[j][2].append(full_set[i][j])
            elif (keys_used[i][2*j:2*j+2] == [1,1]).all():
                sorted_set[j][3].append(full_set[i][j])

    
    return sorted_set


In [ ]:
import h5py

repeats = 10000
session_count = 500

with h5py.File("traces.h5","a") as f:
    for i in range(int(repeats/session_count)):
        print(f"batch_{i}")
        full_set, keys_used = data_collection(session_count, 8)
        target.flush()
        reset_target(scope)
        grp = f.create_group(f"batch_{i}")
        grp.create_dataset("traces", data=np.array(full_set, dtype=np.float32))
        grp.create_dataset("keys", data=np.array(keys_used, dtype=np.int32))

In [ ]:
all_traces = []
all_keys = []

with h5py.File("traces.h5", "r") as f:
    for batch_name in f.keys():   # iterate over batch_0, batch_1, ...
        traces = f[f"{batch_name}/traces"][:]
        keys   = f[f"{batch_name}/keys"][:]
        all_traces.append(traces)
        all_keys.append(keys)

all_traces = np.vstack(all_traces)   # combine into one big array
all_keys   = np.concatenate(all_keys)

In [ ]:
sorted_set = sort_set(session_count, full_set, keys_used, 8)
data_arr = np.array(sorted_set, dtype=object)

In [ ]:
cw.plot(sorted_set[0][0][0]) * cw.plot(sorted_set[0][1][0])

In [ ]:
data_arr = np.array(sorted_set, dtype=object)
np.save("traces.npy", data_arr)

In [ ]:
var_t1bf1_t1bf2 = np.var(np.vstack((data_arr[0][0], data_arr[1][0])),axis=0)
#var_t2bf1_t2bf2 = np.var(np.vstack((t2[:,0], t2[:,1])),axis=0)
#var_t3bf1_t3bf2 = np.var(np.vstack((t3[:,0], t3[:,1])),axis=0)
#var_t4bf1_t4bf2 = np.var(np.vstack((t4[:,0], t4[:,1])),axis=0)

print("T1 BF1-BF2 Var Sum: ", np.sum(var_t1bf1_t1bf2))
#print("T2 BF1-BF2 Var Sum: ", np.sum(var_t2bf1_t2bf2))
#print("T3 BF1-BF2 Var Sum: ", np.sum(var_t3bf1_t3bf2))
#print("T4 BF1-BF2 Var Sum: ", np.sum(var_t4bf1_t4bf2))

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(10,8))
axs = axs.ravel()
titles = ["BF1", "BF2", "BF3", "BF4"]
class_labels = ["00", "01", "10", "11"]

for bf in range(4):
    corr_values = np.zeros((4,4))
    for inpt in range(4):
        for comp in range(4):
            corr_values[inpt,comp] = np.sum(np.var(np.vstack((data_arr[bf][inpt], data_arr[bf][comp])),axis=0)
                                            /(len(data_arr[bf][inpt]) + len(data_arr[bf][comp])))
   
   
    sns.heatmap(corr_values, annot=True, cmap="PuBuGn",
                 xticklabels=class_labels, yticklabels=class_labels, ax=axs[bf])
    axs[bf].set_title(titles[bf])

plt.tight_layout()
plt.show()

